## Working with an LLM programmatically

You have certainly interacted before with a Large Language Model (LLM) like ChatGPT. This is usually done through a UI or an application.

In this Notebook, we are going to use Python to connect and query an LLM directly through its API. For this Lab we are using the **Granite-4.0-H-Tiny** model, served via an external LiteLLM MaaS (Model-as-a-Service) endpoint. This is a fully Open Source model developed by IBM Research.

The model is served externally via a LiteLLM proxy, so no local GPU resources are needed to run this notebook.

### Requirements and Imports

We will then import the libraries we need.

In [ ]:
# Uncomment the following line only if you have not selected the right workbench image, or are using this notebook outside of the workshop environment.
# !pip install --no-cache-dir --no-dependencies --disable-pip-version-check -r requirements.txt

from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain.prompts import PromptTemplate
from langchain_community.llms import VLLMOpenAI

### Langchain

Langchain (https://www.langchain.com/) is a framework for developing applications powered by language models. It will take care for us of all the boilerplate code we would have to manually write to properly query an LLM API.

We will start by creating an **llm** instance, defined by the location where the LLM API can be queried and some parameters that will be applied to the model. For example, `max_tokens` will instruct the model to answer with a maximum of 512 tokens (words or parts of words). `temperature`, set really low here, will instruct the model to stay truth-grounded, and not try to be too "creative". After all, we're not trying to write a fancy poem here!

**IMPORTANT:** Before running the next cell, paste your LiteLLM API token into the `openai_api_key` variable. You can find your token in the lab instructions page.

In [ ]:
# LLM Inference Server URL - External LiteLLM MaaS endpoint
inference_server_url = "https://litellm-prod.apps.maas.redhatworkshops.io"

# LLM definition
llm = VLLMOpenAI(
    openai_api_key="PASTE_YOUR_LITELLM_KEY_HERE",  # Paste your LiteLLM virtual key here
    openai_api_base=f"{inference_server_url}/v1",
    model_name="granite-4-0-h-tiny",
    top_p=0.92,
    temperature=0.01,
    max_tokens=512,
    presence_penalty=1.03,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

We also need a **template** to be applied to every request we are sending to the model (the "Prompt").

When querying a model, you almost never want to send directly what the user has typed. On top of this entry, you need to give proper instructions to the model so that it knows how to handle it: what and how to answer, what NOT to answer, the tone it must use...

In [ ]:
template="""<|system|>
You are a helpful, respectful and honest assistant. Always be as helpful as possible, while being safe.
You will be asked a question, to which you must give an answer.
Your answer should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content.
Please ensure that your responses are socially unbiased and positive in nature.
If a question does not make any sense, or is not factually coherent, explain why instead of answering something not correct.
If you don't know the answer to a question, answer "I don't know".

<|user|>
### QUESTION:
{input}

### ANSWER:
<|assistant|>
"""
prompt = PromptTemplate(input_variables=["input"], template=template)

Langchain allows us to now easily "stitch" those elements together and create a **conversation** object that we will use to query the model.

In [ ]:
conversation = prompt | llm

We are now ready to query the model!

In [ ]:
query = "What is Artificial Intelligence?"

# The streaming callback prints the answer in real-time.
# The final SSE chunk from LiteLLM may contain null text, which triggers
# a Pydantic ValidationError in LangChain's legacy completions parser.
# This is safe to ignore — the full answer has already been streamed.
try:
    conversation.invoke(input=query);
except Exception:
    pass

You can come back to this notebook at section 3.7 for some optional exercises if you want.